# Introdução às Redes Neurais e Deep Learning

## LSTM (Long Short-Term Memory)

A rede do tipo Long Short-term Memory (LSTM)é uma modificação da rede neural recorrente (RNN) padrão destinada a mitigar o problema do desaparecimento do gradiente comumente encontrado pelas RNNs tradicionais. Seu objetivo é fornecer uma memória de curto prazo para RNN que possa durar milhares de passos de tempo (daí o nome "Longa memória de curto prazo").

A rede LSTM acrescenta ao estado oculto $h_t$ das RNNs um novo estado, o estado da célula $C_t$, cuja função é emular uma memória interna de longo prazo carregando informação através do tempo. Além disso são usadas três portas nesta arquitetura:
* Porta de esquecimento $f_t$, que decide o que deve ser descartado do estado da célula no  instante anterior. Valores próximos de 0 são "esquecidos", e valores próximos de 1 "mantidos".
* Porta de entrada: combina duas funções,  $i_t$ decide o montante da nova informação que entra na célula e $\tilde{C}_t$ decide o que deve ser atualizado ao estado da célula.
* Porta de saída $o_t$ decide o quanto do estado atual da célula sera exposto como estado oculto $h_t$.
<div>
<img src="Files/ChatGPT Image LSTM célula.png" width="700" height="600">   
</div>  
No diagrama acima $\sigma$ é a função de ativação sigmoide, $\tanh$ é a tangente hiperbólica, $\otimes$ e $\oplus$ são o produto e adição elemento por elemento. As equações que governam o fluxo de informações nas portas e estados são:
$$ \begin{aligned}
    f_t &= \sigma(W_fz_t+b_f)\\
    i_t &= \sigma(W_iz_t+b_i)\\
    \tilde C_t &= \tanh(W_cz_t+b_c)\\
    o_t &= \sigma(W_oz_t+b_o)\\
    C_t &= f_t\otimes C_{t-1}\oplus i_t\otimes\tilde C_t\\
    h_t &= o_t\otimes\tanh(C_t)
    \end{aligned}$$ 
com $z_t=[h_{t-1},x_t]$.

Um dos principais mecanismos é o controle da atualização do estado da célula $C_t = f_t\otimes C_{t-1}\oplus i_t\otimes\tilde C_t$. Ao longo deste caminho,ou seja, considerando o caminho direto entre os estados da célula, e mantendo fixas as saídas das portas, a derivada local é
$$ \frac{\partial C_t}{\partial C_{t-1}}=f_t,$$
Assim, quando $f_t$ permanece próximo de 1 o gradiente pode ser preservado por longos períodos de tempo, ajudando a mitigar o problema de gradientes evanescentes. 

Uma última porta final de saída (não mostrada acima) é definida como 
$$\hat{y}_t=w_y \cdot h_t +b_y.$$

### Retropropagação pelo tempo (BPTT)

 A retropropagação inicia, como sempre, pela derivada da função de perdas da saída da célula em respeito à própria saída. A derivada depende obviamente da função *Loss* escolhida.  Temos então. inicialmente, $\frac{\partial L}{\partial \hat{y}}$.
 A derivada com respeito aos pesos são obtidas através do método das derivadas parciais (veja as equações de propagação direta na figura acima):

  $$\frac{\partial L}{\partial W_y}= \sum_t \frac{\partial L}{\partial \hat{y}_t} \otimes \frac{\partial \hat{y}_t}{\partial W_y}=\sum_t\frac{\partial L}{\partial \hat{y}_t} \cdot h_t^T$$
  $$\frac{\partial L}{\partial b_y}=\sum_t\frac{\partial L}{\partial \hat{y_t}},$$

Agora, para os outros gradientes, precisaremos primeiro determinar os gradientes em relação ao estado oculto $h_t$ e ao estado da célula $C_t$. Entretanto, é importante notar que em uma LSTM , $h_t$ e $C_t$ influenciam o futuro por múltiplos caminhos simultaneamente. Com efeito, $h_t=o_t \otimes \tanh(C_t)$ e ao mesmo é utilizado em  $h_{t+1}$. Portanto, podemos escrever que

 $$\frac{dL}{dh_t}=\frac{\partial L_t}{\partial h_t}+\left.\frac{dL}{dh_t}\right|_{\text{futuro}}$$
  A primeira parcela representa a contribuição da perda no instante atual, enquanto a segunda representa a contribuição de todas as perdas futuras. Na retropropagação, essas contribuições são acumuladas recursivamente, a partir do instante seguinte. Assim,
  $$\frac{\partial L}{\partial h_t}=\frac{\partial L}{\partial \hat{y}_t}\cdot W_y+\left.\frac{\partial L}{\partial h_t}\right|_{\text{futuro}}$$

Seja $z_t$ a concatenação do estado oculto na saída do passo anterior e da entrada no passo atual, como mostrado nas expressões da figura acima, ou seja, $z_t=\begin{bmatrix}h_{t-1} \\ x_t\end{bmatrix}$. $z_t$ está presente nas definições de $f_t,\;i_t,\;\tilde{c}_t\;\text{e}\;o_t$. Daí que 
$$\frac{\partial L}{\partial z_t}=W_f^T\frac{\partial L}{\partial a_{f,t}}+W_i^T\frac{\partial L}{\partial a_{i,t}}+W_c^T\frac{\partial L}{\partial a_{c,t}}+W_o^T\frac{\partial L}{\partial a_{o,t}}=\begin{bmatrix}\left.\frac{\partial L}{\partial h_{t-1}}\right|_{\text{via}\;z_t}\\ \frac{\partial L}{\partial x_t}\end{bmatrix}$$
onde $a_{f,t},a_{i,t},a_{c,t},a_{o,t}$ são as entradas das respectivas funções de ativação. Por exemplo,
$$\frac{\partial L}{\partial a_{f,t}}=\frac{\partial L}{\partial f_t}\otimes f_t\otimes(1-f_t).$$

Para o estado da célula, $C_t$, temos também que $C_{t+1}=f_{t+1}C_t+ i_{t+1}\tilde{C}_{t+1}$, de forma que além do estado oculto o estado da célula no instante $t+1$ também de pende do estado no instante $t$ (considerando apenas o caminho direto da memória, pois há também a dependência indireta pelas portas), e daí
$$\frac{\partial L}{\partial C_t}=\frac{\partial L}{\partial h_t}\otimes\frac{\partial h_t}{\partial C_t}+\left.\frac{\partial L}{\partial C_t}\right|_{\text{futuro}} ,$$
onde $\left.\frac{\partial L}{\partial C_t}\right|_{\text{futuro}}=f_{t+1}\frac{\partial L}{\partial C_{t+1}}.$  Como $h_t=o_t\otimes\tanh(C_t)$, temos:
$$\frac{\partial h_t}{\partial C_t}=o_t\otimes [1-\tanh^2(C_t)].$$
Portanto 
$$\frac{\partial L}{\partial C_t}=\frac{\partial L}{\partial h_t}\otimes o_t\otimes [1-\tanh^2(C_t)] +f_{t+1}\otimes \frac{\partial L}{\partial C_{t+1}}$$

Os outros gradientes podem ser agora obtidos com facilidade:


$$\frac{\partial L}{\partial W_o}=\sum_t\left(\frac{\partial L}{\partial h_t}\otimes\frac{\partial h_t}{\partial o_t}\otimes\frac{\partial o_t}{\partial W_o}\right)=\sum_t\left[\frac{\partial L}{\partial h_t}\otimes \tanh(C_t)\right]\otimes o_t\otimes(1-o_t)\cdot z_t^T$$

$$\frac{\partial L}{\partial b_o}=\sum_t\left(\frac{\partial L}{\partial h_t}\otimes\frac{\partial h_t}{\partial o_t}\otimes\frac{\partial o_t}{\partial b_o}\right)=\sum_t\left[\frac{\partial L}{\partial h_t}\otimes \tanh(C_t)\right]\otimes o_t\otimes(1-o_t)$$

$$\frac{\partial L}{\partial W_f}=\sum_t\left(\frac{\partial L}{\partial C_t}\otimes\frac{\partial C_t}{\partial f_t}\otimes\frac{\partial f_t}{\partial W_f}\right)=\sum_t\left[\frac{\partial L}{\partial C_t}\otimes C_{t-1}\right]\otimes f_t\otimes(1-f_t)\cdot z_t^T$$

$$\frac{\partial L}{\partial b_f}=\sum_t\left(\frac{\partial L}{\partial C_t}\otimes\frac{\partial C_t}{\partial f_t}\otimes\frac{\partial f_t}{\partial b_f}\right)=\sum_t\left[\frac{\partial L}{\partial C_t}\otimes C_{t-1}\right]\otimes f_t\otimes(1-f_t)$$

$$\frac{\partial L}{\partial W_i}=\sum_t\left(\frac{\partial L}{\partial C_t}\otimes\frac{\partial C_t}{\partial i_t}\otimes\frac{\partial i_t}{\partial W_i}\right)=\sum_t\left[\frac{\partial L}{\partial C_t}\otimes \tilde{C}_t\right]\otimes i_t\otimes(1-i_t)\cdot z_t^T$$

$$\frac{\partial L}{\partial b_i}=\sum_t\left(\frac{\partial L}{\partial C_t}\otimes\frac{\partial C_t}{\partial i_t}\otimes\frac{\partial i_t}{\partial b_i}\right)=\sum_t\left[\frac{\partial L}{\partial C_t}\otimes \tilde{C}_t\right]\otimes i_t\otimes(1-i_t)$$

$$\frac{\partial L}{\partial W_c}=\sum_t\left(\frac{\partial L}{\partial C_t}\otimes\frac{\partial C_t}{\partial \tilde{C}_t}\otimes\frac{\partial \tilde{C}_t}{\partial W_c}\right)=\sum_t\left[\frac{\partial L}{\partial C_t}\otimes i_t\right]\otimes (1-\tilde{C}_t^2)\cdot z_t^T$$

$$\frac{\partial L}{\partial b_c}=\sum_t\left(\frac{\partial L}{\partial C_t}\otimes\frac{\partial C_t}{\partial \tilde{C}_t}\otimes\frac{\partial \tilde{C}_t}{\partial b_c}\right)=\sum_t\left[\frac{\partial L}{\partial C_t}\otimes i_t\right]\otimes (1-\tilde{C}_t^2)$$

### Implementação de uma LSTM mínima em NumPy.
Como exemplo inicial, implementaremos o processo de *feedforward/backpropagation* em Python/NumPY simulando duas etapas temporais, a backpropagation e a atualização dos pesos. Para simplificar o exemplo numérico, consideraremos a saída da LSTM diretamente como a predição, isto é, $\hat y_2=h_2$. Dessa forma, não será necessário introduzir os parâmetros $W_y$ e $b_y$.

#### Classe LSTM implementada em Python/Numpy

In [1]:
import numpy as np

class LSTM:
    def __init__(self, lr=0.1):

        self.lr = lr
        # pesos
        self.Wf = np.array([[0.2, 0.7]])
        self.Wi = np.array([[0.1, 0.5]])
        self.Wo = np.array([[0.3, 0.9]])
        self.Wc = np.array([[0.2, 0.6]])

        self.bf = np.array([[0.1]])
        self.bi = np.array([[0.1]])
        self.bo = np.array([[0.1]])
        self.bc = np.array([[0.1]])

        self.params = ["Wf", "Wi", "Wo", "Wc",
                       "bf", "bi", "bo", "bc"]

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    # ======================================
    # FORWARD
    # ======================================
    def forward(self, X):

        h = np.zeros((1, 1))
        c = np.zeros((1, 1))
        
        self.cache = []

        for x in X:

            x = np.array([[x]])

            # Concatenação [h_(t-1),x_t]
            z = np.vstack((h,x))

            f = self.sigmoid(self.Wf @ z + self.bf)
            i = self.sigmoid(self.Wi @ z + self.bi)
            o = self.sigmoid(self.Wo @ z + self.bo)

            c_til = np.tanh(self.Wc @ z + self.bc)

            c_new = f * c + i * c_til
            h_new = o * np.tanh(c_new)

            # Armazenar valores para o BPTT
            self.cache.append({"z": z,"f": f,"i": i,"o": o,"c_til": c_til,"c_prev": c,"c": c_new,"h": h_new})

            h = h_new
            c = c_new

        return h

    # ======================================
    # BACKWARD THROUGH TIME
    # ======================================

    def backward(self, target):
        # Inicialização dos gradientes
        self.grads = {p: np.zeros_like(getattr(self, p)) for p in self.params }

        h_final = self.cache[-1]["h"]

        # Loss = (h_final - target)^2
        loss = (h_final - target)**2

        # Derivada da perda no último instante
        dh_next = 2 * (h_final - target)

        # Não há perda futura após o último instante
        dc_next = np.zeros((1, 1))

        # Percorrer os instantes em ordem inversa
        for t in reversed(range(len(self.cache))):

            dados = self.cache[t]

            z = dados["z"]
            f = dados["f"]
            i = dados["i"]
            o = dados["o"]
            c_til = dados["c_til"]
            c_prev = dados["c_prev"]
            c = dados["c"]

            # Gradiente da porta de saída
            do = dh_next * np.tanh(c)
            delta_o = do * o * (1 - o)

            # Gradiente total do estado da célula
            dc = (dh_next * o * (1 - np.tanh(c)**2)+ dc_next)

            # Porta de esquecimento
            df = dc * c_prev
            delta_f = df * f * (1 - f)

            # Porta de entrada
            di = dc * c_til
            delta_i = di * i * (1 - i)

            # Estado candidato
            dc_til = dc * i
            delta_c = dc_til * (1 - c_til**2)

            # Gradientes dos pesos
            self.grads["Wf"] += delta_f @ z.T
            self.grads["Wi"] += delta_i @ z.T
            self.grads["Wo"] += delta_o @ z.T
            self.grads["Wc"] += delta_c @ z.T

            # Gradientes dos bias
            self.grads["bf"] += delta_f
            self.grads["bi"] += delta_i
            self.grads["bo"] += delta_o
            self.grads["bc"] += delta_c

            # Gradiente em relação à concatenação z
            dz = (self.Wf.T @ delta_f + self.Wi.T @ delta_i + self.Wo.T @ delta_o + self.Wc.T @ delta_c)
            
            # Primeira componente: gradiente futuro de h (z = [h_(t-1), x_t)
            dh_next = dz[0:1, :]

            # Caminho direto da memória
            dc_next = dc * f

        return loss.item()

    # ======================================
    # ATUALIZAÇÃO DOS PESOS
    # ======================================
    def update(self):

        for p in self.params:
            valor = getattr(self, p)
            setattr(self, p,valor - self.lr * self.grads[p])

    # ======================================
    # EXIBIÇÃO DOS PARÂMETROS
    # ======================================
    def show_parameters(self):

        for p in self.params:
            print(f"{p} = {getattr(self, p)}")


#### Execução

In [2]:

# Dados de entrada
X = [0.5, 0.8]
target = 0.4

# Criar a LSTM
model = LSTM(lr=0.1)

# Forward
h2 = model.forward(X)

print("======== FORWARD ========")

for t, dados in enumerate(model.cache, start=1):
    print(f"\nTempo {t}")
    for nome in ["f", "i", "o", "c_til", "c", "h"]:
        print(f"{nome} = {dados[nome].item():.8f}")

# Backward
loss = model.backward(target)

print("\n======== LOSS ========")
print("Loss =", loss)

print("\n======== GRADIENTES ========")

for p, grad in model.grads.items():
    print(f"{p} = {grad}")

# Atualização
model.update()

print("\n======== PESOS ATUALIZADOS ========")
model.show_parameters()


======== FORWARD ========

Tempo 1
f = 0.61063923
i = 0.58661758
o = 0.63413559
c_til = 0.37994896
c = 0.22288474
h = 0.13904427

Tempo 2
f = 0.66547932
i = 0.62572132
o = 0.70301848
c_til = 0.54258286
c = 0.48783084
h = 0.31811102

======== LOSS ========
Loss = 0.006705805491948991

======== GRADIENTES ========
Wf = [[-0.00063171 -0.00363458]]
Wi = [[-0.00161779 -0.01252672]]
Wo = [[-0.00215138 -0.01275452]]
Wc = [[-0.00562109 -0.04987575]]
bf = [[-0.00454322]]
bi = [[-0.01807241]]
bo = [[-0.01622547]]
bc = [[-0.07549551]]

======== PESOS ATUALIZADOS ========
Wf = [[0.20006317 0.70036346]]
Wi = [[0.10016178 0.50125267]]
Wo = [[0.30021514 0.90127545]]
Wc = [[0.20056211 0.60498758]]
bf = [[0.10045432]]
bi = [[0.10180724]]
bo = [[0.10162255]]
bc = [[0.10754955]]


### Geração de texto usando LSTM a partir de um *dataset* formado por um conjunto  diferentes poemas

#### Importação da bibliotecas

In [13]:
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

import pandas as pd
import numpy as np
import random

#### Leitura do *dataset*

In [4]:
df = pd.read_csv('Files/poems.csv', header=None, names=['text'])
text = " ".join(df['text'].tolist())

print("Total de poemas:", len(df))
print("Poemas diferentes:", df['text'].nunique())

print("\nPoemas mais frequentes:")
print(df['text'].value_counts().head(10))

Total de poemas: 15000
Poemas diferentes: 15

Poemas mais frequentes:
text
A single leaf falls, carried away by the gentle breeze.                          1072
A whisper of hope in the silence of the dawn.                                    1042
The sun sets in the sky, painting the horizon with a crimson hue.                1038
In the stillness of the garden, flowers bloom with grace.                        1028
Love is the light that guides us through the darkest times.                      1012
In the stillness of night, the moon whispers its light.                          1005
Stars above, guiding me through the endless sea of dreams.                       1002
Roses are red, violets are blue, sugar is sweet, and so are you.                  998
In the heart of winter, snowflakes dance and play.                                991
Through the forest deep, where shadows linger long, the night sings its song.     986
Name: count, dtype: int64


#### vocab: conjunto de caracteres (tokens) únicos na base de dados.

In [5]:
vocab = sorted(set(text))
char_to_index = {c: i for i, c in enumerate(vocab)}
index_to_char = {i: c for i, c in enumerate(vocab)}
print(vocab)

[' ', ',', '.', 'A', 'B', 'I', 'L', 'R', 'S', 'T', 'U', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z']


#### Criação de Sequências de Comprimento Fixo e dos respectivos Rótulos, codificação da entrada e do rótulos em vetores One-Hot:

In [6]:
seq_length = 100
step = 5

sequences = []
labels = []

for i in range(0, len(text) - seq_length, step):
    seq = text[i:i + seq_length]
    label = text[i + seq_length]

    sequences.append([char_to_index[c] for c in seq])
    labels.append(char_to_index[label])

X = np.array(sequences)
y = np.array(labels)

X = to_categorical(X, num_classes=len(vocab))
y = to_categorical(y, num_classes=len(vocab))

#### Criação do  modelo LSTM 

In [7]:
model = Sequential([
    Input(shape=(seq_length,len(vocab))),
    LSTM(128,activation='tanh'),
    Dense(len(vocab), activation='softmax')
])

optimizer = Adam(learning_rate=0.001)
model.compile(loss='categorical_crossentropy', optimizer=optimizer)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 128)                 │          83,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 35)                  │           4,515 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 88,483 (345.64 KB)

 Trainable params: 88,483 (345.64 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.fit(X, y,batch_size=128,epochs=30)

Epoch 1/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 122s 87ms/step - loss: 2.1619
Epoch 2/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 119s 85ms/step - loss: 0.1646
Epoch 3/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 118s 85ms/step - loss: 0.0646
Epoch 4/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 117s 85ms/step - loss: 0.0811
Epoch 5/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 121s 87ms/step - loss: 0.0531
Epoch 6/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 120s 87ms/step - loss: 0.0524
Epoch 7/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 120s 87ms/step - loss: 0.0477
Epoch 8/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 121s 87ms/step - loss: 0.0484
Epoch 9/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 120s 86ms/step - loss: 0.0540
Epoch 10/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 112s 81ms/step - loss: 0.0481
Epoch 11/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 104s 75ms/step - loss: 0.0456
Epoch 12/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 104s 75ms/step - loss: 0.0457
Epoch 13/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 105s 76ms/step - loss: 0.0470
Epoch 14/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 104s 75ms/step - 

Durante o treinamento, a rede aprendeu a associação:
$$\underbrace{(c_1,c_2,\ldots,c_{100})}_{\text{100 caracteres de entrada}}\quad\longrightarrow\quad
\underbrace{P(c_{101})}_{\text{probabilidades do próximo caractere}}$$

A saída da rede não é diretamente um caractere. É um vetor com 35 probabilidades, uma para cada símbolo do vocabulário.

Durante a geração, o seguinte procedimento é repetido:
<div>
<img src="Files/Fluxo_Ger_Texto.png" width="400" height="400">   
</div>  

A janela sempre terá 100 caracteres. O texto gerado, entretanto, cresce a cada iteração.

* Função `sample`:ajusta as probabilidades de saída do modelo usando a temperatura para controlar a aleatoriedade e, em seguida, amostra o próximo índice de caractere a partir da  nova distribuição.

  `preds = np.asarray(preds).astype('float64')`: `preds` é o vetor de probabilidades produzido pela camada Dense com softmax.Se a temperatura é zero, o índice de maior valor é escolhido.

  `preds = np.log(preds + 1e-8) / temperature`: se as probabilidades originais forem $p_i$, então
  $$z_i=\frac{\ln(p_i+\epsilon)}{T}$$

  onde $T$ é a temperatura e $\epsilon=10^{-8}$ garante que não ocorra $\ln(0)$. Calculando novamente a softmax:
  
  $$q_i=\frac{e^{z_i}}{\sum_j e^{z_j}}=\frac{p_i^{1/T}}{\sum_j p_j^{1/T}}$$
  Onde $\epsilon$ foi desprezado. como resultado temos uma nova distribuição de probabilidades, mantendo a sua ordem original. om $T=1$, as probabilidades originais são preservadas. Com $T<1$, favorece-se os caracteres mais prováveis. Com $T>1$, a distribuição se torna mais uniforme.

  `exp_preds = np.exp(preds - np.max(preds))`: calcula as esponenciais onde `np.max(preds)` é subtraída no numerador e denominador  para melhorar a estabilidade numérica.
  `preds = exp_preds / np.sum(exp_preds)`: normalização dos valores, de forma que sua soma seja igual a 1.
  'probas = np.random.multinomial(1, preds, 1)`: a função `multinomial()`realiza um sorteio de acordo com as probabilidades fornecidas (compare com a função `argmax()`usada anteriormente, sem temperatura. Como resultado a função retorna em cada posição o número de vezes que cada índice repectivo foi sorteado.
  
  `np.argmax(probas)`: identifica a posição do 1, retornando o índice do caractere sorteado.

  * Função `generate_text()`: iniciando com uma sequência inicial aleatória, prevê repetidamente o próximo caractere usando o modelo e a função `sample()`, atualiza a sequência de entrada e constrói um texto gerado com o comprimento especificado.
    
       `start_idx = random.randint(0, len(text) - seq_length  - 1)`: escolhe aleatoriamente uma posição do texto original. `sentences ` será uma lista com os 100 caracteres do conjunto de treino a partir da posição escolhida. Isto forma o contexto original e é denominado *seed*.

       `x_pred = np.zeros((1, seq_length, len(vocab)))`: `x_pred.shape` = (1, 100, 35), uma única sequência, 100 caracteres da janela e 35 símbolos do vocabulário. 


In [11]:
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    if temperature == 0:
        return np.argmax(preds)
    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds - np.max(preds))
    preds = exp_preds / np.sum(exp_preds)

    probas = np.random.multinomial(1, preds, 1) #1 sorteio, retorna um conjunto de resultados nulos e 1 para o índice sorteado

    return np.argmax(probas)

def generate_text(length=400, temperature=0.5):
    start_idx = random.randint(0, len(text) - seq_length  - 1)
    generated = ''
    sentence = text[start_idx: start_idx + seq_length ]
    generated += sentence

    for _ in range(length):
        x_pred = np.zeros((1, seq_length , len(vocab)))
        for t, char in enumerate(sentence):
            x_pred[0, t, char_to_index[char]] = 1

        preds = model.predict(x_pred, verbose=0)[0]
        next_idx = sample(preds, temperature)
        next_char = index_to_char[next_idx]

        generated += next_char
        sentence = sentence[1:] + next_char

    return generated

In [12]:
print(generate_text(length=500, temperature=0.7))

 song. Roses are red, violets are blue, sugar is sweet, and so are you. Stars above, guiding me through the endless sea of dreams. The sun sets in the sky, painting the horizon with a crimson hue. The sun sets in the sky, painting the horizon with a crimson hue. Roses are red, violets are blue, sugar is sweet, and so are you. A single leaf falls, carried away by the gentle breeze. In the heart of winter, snowflakes dance and play. The mountain peaks kiss the sky, standing proud and wise. In the stillness of the garden, flowers bloom with grace. In the stillness of the garden, flowers bloom wit


Na geração de texto por LSTM, a escolha determinística do próximo caractere (temperatura zero) tende a produzir sequências mais previsíveis e, eventualmente, periódicas. A amostragem probabilística permite explorar diferentes continuações, aumentando a diversidade, mas também a possibilidade de gerar sequências incorretas. Portanto, existe um compromisso entre previsibilidade e diversidade na geração.